Sentiment Analysis 

In [26]:
import pandas as pd
import re
from collections import Counter

## Function to import the dict

In [36]:
def load_filtered_liwc_dictionary(filepath, allowed_ids):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    category_map = {}
    liwc_dict = {}
    percent_count = 0

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line == '%':
            percent_count += 1
            continue

        parts = line.split()

        # Kategorie-Definitionen
        if percent_count == 1:
            if len(parts) >= 2:
                cat_id = parts[0]
                cat_name = parts[1]
                if cat_id in allowed_ids:
                    category_map[cat_id] = cat_name

        # Wort-Kategorien
        elif percent_count == 2:
            word = parts[0]
            cat_ids = parts[1:]
            # Nur erlaubte Kategorien
            filtered_ids = [cid for cid in cat_ids if cid in allowed_ids]
            categories = [category_map.get(cid) for cid in filtered_ids if cid in category_map]
            if categories:
                liwc_dict[word] = categories

    return liwc_dict


When comparing the studies in our literature review, I decided to shrink the different LIWC Categories to seven most relevant. 

01	Pronoun, 13	Positiveemotion, 16	Negativeemotion, 35	Family, 50	Achieve, 58	Relig, 15	Optimism.

In [ ]:
allowed_ids = {'01', '13', '15', '16', '35', '50', '58'}
liwc_dict = load_filtered_liwc_dictionary("../data/LIWC_German.txt", allowed_ids)

### After converting the dictionary, the following includes a function to use the dictionary on datasets

In [39]:
def analyze_text_liwc(text, liwc_dict):
    if pd.isnull(text):
        return {}
    
    tokens = re.findall(r'\b\w+\b', text.lower())
    token_count = len(tokens)
    if token_count == 0:
        return {}

    category_counts = {}

    for token in tokens:
        for word, categories in liwc_dict.items():
            if word.endswith("*"):
                if token.startswith(word[:-1]):
                    for cat in categories:
                        category_counts[cat] = category_counts.get(cat, 0) + 1
            elif token == word:
                for cat in categories:
                    category_counts[cat] = category_counts.get(cat, 0) + 1

    # Umwandlung in Prozentwerte
    category_percentages = {cat: (count / token_count) * 100 for cat, count in category_counts.items()}
    return category_percentages


In [47]:
liwc_dict = load_filtered_liwc_dictionary("../data/LIWC_German.txt", allowed_ids)


   party        topic  Positiveemotion  Family  Negativeemotion  Achieve  \
0    AfD  Mindestlohn             0.00    0.00             0.00     0.00   
1  Union  Mindestlohn             6.19    0.44             1.62     9.14   
2  LINKE  Mindestlohn             6.04    0.95             1.75     6.04   
3  GRÜNE  Mindestlohn             6.48    0.69             1.24     6.76   
4    FDP  Mindestlohn             6.42    0.17             2.53     5.74   

   Relig  Optimism  
0   0.00      0.00  
1   0.29      1.47  
2   0.48      1.91  
3   0.28      1.52  
4   0.17      1.18  


C:\Users\Pasca\AppData\Local\Temp\ipykernel_27516\3810863185.py:8: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  liwc_df = data_got["liwc"].apply(pd.Series).fillna(0)
C:\Users\Pasca\AppData\Local\Temp\ipykernel_27516\3810863185.py:8: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  liwc_df = data_got["liwc"].apply(pd.Series).fillna(0)


#### Data from gpt prompt

In [40]:
data_got = pd.read_csv("../data/gpt_speeches/gpt_speeches.csv", sep=";")  

In [41]:
data_got["liwc_counts"] = data_got["speech_text"].apply(lambda x: analyze_text_liwc(x, liwc_dict))

In [ ]:
data_got["liwc"] = data_got["speech_text"].apply(lambda x: analyze_text_liwc(x, liwc_dict))
liwc_df = data_got["liwc"].apply(pd.Series).fillna(0)
final_df = pd.concat([data_got[["party", "topic"]], liwc_df], axis=1)
final_df = final_df.round(2)
print(final_df.head())

In [ ]:
final_df.head()

,party,topic,Positiveemotion,Family,Negativeemotion,Achieve,Relig,Optimism
0,AfD,Mindestlohn,0.00,0.00,0.00,0.00,0.00,0.00
1,Union,Mindestlohn,6.19,0.44,1.62,9.14,0.29,1.47
2,LINKE,Mindestlohn,6.04,0.95,1.75,6.04,0.48,1.91
3,GRÜNE,Mindestlohn,6.48,0.69,1.24,6.76,0.28,1.52
4,FDP,Mindestlohn,6.42,0.17,2.53,5.74,0.17,1.18


In [54]:
dataOwn.head(1)

,party,topic,speech_base_model_01,speech_ft_model_01,speech_base_model_03,speech_ft_model_03,RAG_speech_01,context_speeches_x,RAG_speech_03,context_speeches_y
0,Union,Mindestlohn,#HartzIV - Die Zeit der Verantwortung für die ...,[INST] Herr Präsident! Liebe Kolleginnen und l...,#HartzIV - Die Zeit der Verantwortung für die ...,[Inst] Frau Präsidentin! Liebe Kolleginnen und...,[INKLUSIVE SPEICHERUNGEBENE]. Herr Präsident! ...,NaN,[EINSETZ DER SICHERHEITSBEREITSTELLEN DES BUND...,NaN


### Own generated speeches: Data cleanining

In [68]:
dataOwn= pd.read_csv("../data/generated_speeches_final_context_V2.csv")

In [69]:
dataOwn = dataOwn.drop(columns=["context_speeches_x", "context_speeches_y"])

In [ ]:
string_cols = dataOwn.select_dtypes(include=["object", "string"]).columns
important_cols = [c for c in string_cols if c not in ["party", "topic"]]

dataOwn[important_cols] = dataOwn[important_cols].apply(
    lambda s: (
        s.astype("string")  
         .str.replace(r"#", "", regex=True)
         .str.replace(r"inst", "", regex=True, flags=re.IGNORECASE)
         .str.replace(r"\[\]", "", regex=True)  
    )
)

In [72]:
dataOwn.head(1)

,party,topic,speech_base_model_01,speech_ft_model_01,speech_base_model_03,speech_ft_model_03,RAG_speech_01,RAG_speech_03
0,Union,Mindestlohn,HartzIV - Die Zeit der Verantwortung für die A...,Herr Präsident! Liebe Kolleginnen und lieber ...,HartzIV - Die Zeit der Verantwortung für die A...,Frau Präsidentin! Liebe Kolleginnen und liebe...,[INKLUSIVE SPEICHERUNGEBENE]. Herr Präsident! ...,[EINSETZ DER SICHERHEITSBEREITSTELLEN DES BUND...


In [73]:
dataOwn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   party                 24 non-null     object
 1   topic                 24 non-null     object
 2   speech_base_model_01  24 non-null     string
 3   speech_ft_model_01    24 non-null     string
 4   speech_base_model_03  24 non-null     string
 5   speech_ft_model_03    24 non-null     string
 6   RAG_speech_01         24 non-null     string
 7   RAG_speech_03         24 non-null     string
dtypes: object(2), string(6)
memory usage: 1.6+ KB


In [ ]:
no_col_text  = [c for c in string_cols if c not in ["party", "topic"]]

In [ ]:
text_cols = [c for c in dataOwn.columns if c not in ["party", "topic"]]

# 2) Alle Kategorien, die im Dictionary existieren (für sauberes Reindexing)
all_cats = sorted({cat for cats in liwc_dict.values() for cat in cats})

# 3) Pro Textspalte analysieren und andocken
for col in text_cols:
    out = (dataOwn[col]
           .apply(lambda x: analyze_text_liwc(x, liwc_dict))  # dict je Zeile
           .apply(pd.Series)                                  # zu Spalten
           .reindex(columns=all_cats)                         # fehlende Cats = NaN
           .fillna(0.0))                                      # -> 0
    out.columns = [f"{col}__LIWC_{c}" for c in out.columns]   # sprechende Namen
    dataOwn = pd.concat([dataOwn, out], axis=1)

In [74]:
import re
import pandas as pd

# 1) Textspalten bestimmen (alles außer party/topic)
text_cols = [c for c in dataOwn.columns if c not in ["party", "topic"]]

# 2) Alle (gefilterten) Kategorien aus dem liwc_dict sammeln
cats = sorted({cat for cats_ in liwc_dict.values() for cat in cats_})

# 3) Deine LIWC-Funktion (leicht robust gegen Nicht-Strings)
def analyze_text_liwc(text, liwc_dict):
    if pd.isnull(text):
        return {}
    tokens = re.findall(r'\b\w+\b', str(text).lower())
    token_count = len(tokens)
    if token_count == 0:
        return {}
    category_counts = {}
    for token in tokens:
        for word, categories in liwc_dict.items():
            if word.endswith("*"):
                if token.startswith(word[:-1]):
                    for cat in categories:
                        category_counts[cat] = category_counts.get(cat, 0) + 1
            elif token == word:
                for cat in categories:
                    category_counts[cat] = category_counts.get(cat, 0) + 1
    return {cat: (count / token_count) * 100 for cat, count in category_counts.items()}

# 4) Helper: eine Model-Spalte analysieren
def liwc_for_col(col):
    out = (dataOwn[col]
           .apply(lambda x: analyze_text_liwc(x, liwc_dict))
           .apply(pd.Series)
           .reindex(columns=cats)
           .fillna(0.0))
    out.insert(0, "party", dataOwn["party"].values)
    out.insert(1, "topic", dataOwn["topic"].values)
    out.insert(2, "model", col)
    return out

# 5) Alle Model-Spalten verarbeiten und zusammenführen
frames = [liwc_for_col(col) for col in text_cols]
data_liwc_own = pd.concat(frames, ignore_index=True)

# 6) Falls mehrere Zeilen je (party, topic, model) existieren → mitteln & runden
data_liwc_own = (data_liwc_own
                 .groupby(["party", "topic", "model"], as_index=False)[cats]
                 .mean()
                 .round(2))

print(data_liwc_own.head())


C:\Users\Pasca\AppData\Local\Temp\ipykernel_27516\38460693.py:34: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  .apply(pd.Series)
C:\Users\Pasca\AppData\Local\Temp\ipykernel_27516\38460693.py:34: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  .apply(pd.Series)
C:\Users\Pasca\AppData\Local\Temp\ipykernel_27516\38460693.py:34: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  .apply(pd.Series)
C:\Users\Pasca\AppData\Local\Temp\ipykernel_27516\38460693.py:34: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  .apply(pd.Series)
C:\Users\Pas

  party                        topic                 model  Achieve  Family  \
0   AfD  Bundeswehreinsatz im Kosovo         RAG_speech_01     4.99    0.96   
1   AfD  Bundeswehreinsatz im Kosovo         RAG_speech_03     2.92    1.95   
2   AfD  Bundeswehreinsatz im Kosovo  speech_base_model_01     6.34    0.00   
3   AfD  Bundeswehreinsatz im Kosovo  speech_base_model_03     1.96    0.73   
4   AfD  Bundeswehreinsatz im Kosovo    speech_ft_model_01     3.19    0.56   

   Negativeemotion  Optimism  Positiveemotion  Relig  
0             1.34      1.54             4.99   1.73  
1             1.39      0.56             1.53   0.56  
2             2.05      2.24             7.46   1.31  
3             2.20      0.49             1.71   0.73  
4             1.31      0.75             1.97   1.41  


In [ ]:
data_liwc_own_long = data_liwc_own.melt(
    id_vars=["party", "topic", "model"],
    value_vars=cats,
    var_name="liwc_category",
    value_name="percent"
)


,party,topic,model,liwc_category,percent
0,AfD,Bundeswehreinsatz im Kosovo,RAG_speech_01,Achieve,4.99
1,AfD,Bundeswehreinsatz im Kosovo,RAG_speech_03,Achieve,2.92
2,AfD,Bundeswehreinsatz im Kosovo,speech_base_model_01,Achieve,6.34
3,AfD,Bundeswehreinsatz im Kosovo,speech_base_model_03,Achieve,1.96
4,AfD,Bundeswehreinsatz im Kosovo,speech_ft_model_01,Achieve,3.19


In [77]:
data_liwc_own_long.head(60)

,party,topic,model,liwc_category,percent
0,AfD,Bundeswehreinsatz im Kosovo,RAG_speech_01,Achieve,4.99
1,AfD,Bundeswehreinsatz im Kosovo,RAG_speech_03,Achieve,2.92
2,AfD,Bundeswehreinsatz im Kosovo,speech_base_model_01,Achieve,6.34
3,AfD,Bundeswehreinsatz im Kosovo,speech_base_model_03,Achieve,1.96
4,AfD,Bundeswehreinsatz im Kosovo,speech_ft_model_01,Achieve,3.19
5,AfD,Bundeswehreinsatz im Kosovo,speech_ft_model_03,Achieve,3.73
6,AfD,Gaspreise,RAG_speech_01,Achieve,3.92
7,AfD,Gaspreise,RAG_speech_03,Achieve,7.04
8,AfD,Gaspreise,speech_base_model_01,Achieve,14.35
9,AfD,Gaspreise,speech_base_model_03,Achieve,5.51
